In [27]:
import os
os.chdir("/Users/nicoleperez/Desktop/PCHFINAL")

In [28]:
import json

with open("sites.json", "r") as f:
    data = json.load(f)

print(type(data))
print(str(data)[:500])

<class 'list'>
[{'id': 2065, 'date': '2021-06-24T10:50:42', 'date_gmt': '2021-06-24T10:50:42', 'guid': {'rendered': 'https://www.nyclgbtsites.org/?post_type=sites&#038;p=2065'}, 'modified': '2022-08-20T10:55:22', 'modified_gmt': '2022-08-20T14:55:22', 'slug': 'new-york-public-library-135th-street-library', 'status': 'publish', 'type': 'sites', 'link': 'https://www.nyclgbtsites.org/site/new-york-public-library-135th-street-library/', 'title': {'rendered': '135th Street Branch, New York Public Library'}, 'author


In [29]:
print(data[0].keys())
print()
print(data[0])

dict_keys(['id', 'date', 'date_gmt', 'guid', 'modified', 'modified_gmt', 'slug', 'status', 'type', 'link', 'title', 'author', 'featured_media', 'template', 'meta', 'cultural_significance', 'neighborhood', 'era', 'lgbt_category', 'site_type', 'class_list', 'site_type_color', 'image_url', 'site_neighborhood', 'featured_image_by_size', 'acf', '_links'])

{'id': 2065, 'date': '2021-06-24T10:50:42', 'date_gmt': '2021-06-24T10:50:42', 'guid': {'rendered': 'https://www.nyclgbtsites.org/?post_type=sites&#038;p=2065'}, 'modified': '2022-08-20T10:55:22', 'modified_gmt': '2022-08-20T14:55:22', 'slug': 'new-york-public-library-135th-street-library', 'status': 'publish', 'type': 'sites', 'link': 'https://www.nyclgbtsites.org/site/new-york-public-library-135th-street-library/', 'title': {'rendered': '135th Street Branch, New York Public Library'}, 'author': 3, 'featured_media': 2066, 'template': '', 'meta': {'_acf_changed': False, '_monsterinsights_skip_tracking': False, '_monsterinsights_sitenote_a

In [30]:
!pip install folium

In [31]:
import json
import os
import folium
from folium.plugins import MarkerCluster

In [32]:
os.chdir("/Users/nicoleperez/Desktop/PCHFINAL")

In [33]:
with open("sites.json", "r") as f:
    data = json.load(f)

print(f"Total sites in file: {len(data)}")

Total sites in file: 511


In [34]:
target_eras = {"era-1970s", "era-1980s", "era-1990s"}

allowed_types = {
    "Bars, Clubs & Restaurants",
    "Organization & Community Spaces",
    "Performance Venues",
}

print(f"Filtering for: {sorted(target_eras)}")
print(f"Allowed site types: {sorted(allowed_types)}")

Filtering for: ['era-1970s', 'era-1980s', 'era-1990s']
Allowed site types: ['Bars, Clubs & Restaurants', 'Organization & Community Spaces', 'Performance Venues']


In [45]:
results = []

for site in data:
    class_list = site.get("class_list", [])

    site_eras = set(class_list) & target_eras
    if not site_eras:
        continue

    if any("cruising" in c.lower() for c in class_list):
        continue

    site_type = site.get("site_type", {}).get("name", "")
    if site_type not in allowed_types:
        continue

    lat = site.get("meta", {}).get("latitude", "")
    lon = site.get("meta", {}).get("longitude", "")
    if not lat or not lon:
        continue

    try:
        lat = float(lat)
        lon = float(lon)
    except:
        continue

    name = site.get("title", {}).get("rendered", "Unknown")
    address = site.get("meta", {}).get("site_location", "")
    url = site.get("link", "")
    neighborhood = site.get("site_neighborhood", "")
    image = site.get("image_url", "")
    eras = [c.replace("era-", "") for c in class_list if c.startswith("era-") and c in target_eras]

    results.append({
        "name": name,
        "address": address,
        "lat": lat,
        "lon": lon,
        "url": url,
        "neighborhood": neighborhood,
        "site_type": site_type,
        "image": image,
        "eras": ", ".join(eras),
    })

print(f"Sites after filtering: {len(results)}")

Sites after filtering: 61


In [36]:
type_colors = {
    "Bars, Clubs & Restaurants": "red",
    "Organization & Community Spaces": "orange",
    "Performance Venues": "pink",
}

def get_color(site_type):
    return type_colors.get(site_type, "gray")

In [37]:
def build_popup(site):
    image_html = ""
    if site["image"]:
        image_html = f'<img src="{site["image"]}" width="260px" style="margin-bottom:8px; border-radius:6px;"><br>'

    return f"""
        {image_html}
        <b style="font-size:14px;">{site['name']}</b><br>
        <span style="color:#aaa;">{site['address']}</span><br>
        <span style="color:#aaa;">{site['neighborhood']}</span><br><br>
        <b>Type:</b> {site['site_type']}<br>
        <b>Era:</b> {site['eras']}<br><br>
        <a href="{site['url']}" target="_blank"
           style="background:#6a0dad; color:white; padding:4px 10px;
                  border-radius:4px; text-decoration:none;">
           View Site →
        </a>
    """

In [38]:
m = folium.Map(
    location=[40.7300, -73.9900],
    zoom_start=12,
    tiles="CartoDB dark_matter"
)

cluster_70s = MarkerCluster(name="1970s").add_to(m)
cluster_80s = MarkerCluster(name="1980s").add_to(m)
cluster_90s = MarkerCluster(name="1990s").add_to(m)

print("Base map and decade layers created")

Base map and decade layers created


In [39]:
for site in results:
    color = get_color(site["site_type"])
    popup_html = build_popup(site)

    def make_marker(site, color, popup_html):
        return folium.Marker(
            location=[site["lat"], site["lon"]],
            popup=folium.Popup(popup_html, max_width=300),
            tooltip=f"{site['name']} ({site['site_type']})",
            icon=folium.Icon(color=color, icon="info-sign")
        )

    if "1970s" in site["eras"]:
        make_marker(site, color, popup_html).add_to(cluster_70s)

    if "1980s" in site["eras"]:
        make_marker(site, color, popup_html).add_to(cluster_80s)

    if "1990s" in site["eras"]:
        make_marker(site, color, popup_html).add_to(cluster_90s)

print(f"{len(results)} sites added to map")

61 sites added to map


In [40]:
folium.LayerControl(collapsed=False).add_to(m)

print("Layer control added")

Layer control added


In [43]:
legend_html = """
<div style="position: fixed; bottom: 40px; left: 40px; z-index: 1000;
            background: rgba(0,0,0,0.75); padding: 14px 18px;
            border-radius: 8px; color: white; font-size: 13px;">
    <b style="font-size:14px;">Site Type</b><br><br>
    <span style="color:#d43f3a;">●</span> Bars, Clubs & Restaurants<br>
    <span style="color:#f69730;">●</span> Organization & Community<br>
    <span style="color:#ff69b4;">●</span> Performance Venues<br>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

print("Success")

Success


In [44]:
m